# 实验 3 & 5 — 多智能体验证 + MMDFND 复现
## CDS525 Group Project

| 实验 | 方法 | 预计耗时 |
|------|------|:--------:|
| 组3 | 多智能体：Claim → 网搜 → NLI → Judge | ~5-15 min |
| 组5 | MMDFND 论文复现（中文微博多模态虚假新闻检测） | ~3-5 h |

**运行前**：Runtime → Change runtime type → **GPU (推荐 T4 或更高)**

---
## 0. 环境配置

In [ ]:
# ============================================================
# 0.1 检查 GPU
# ============================================================
import torch, os, sys, time, shutil

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> GPU")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ============================================================
# 0.2 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/fakenews-detector'
DRIVE_DATA3   = '/content/drive/MyDrive/data 3'   # 实验5 微博 CSV

print(f"项目目录: {DRIVE_PROJECT}  存在: {os.path.isdir(DRIVE_PROJECT)}")
print(f"data 3 目录: {DRIVE_DATA3}  存在: {os.path.isdir(DRIVE_DATA3)}")

In [ ]:
# ============================================================
# 0.3 复制项目到本地磁盘 (Drive I/O 太慢)
# ============================================================
PROJECT_DIR = '/content/fakenews-detector'

if not os.path.exists(PROJECT_DIR):
    print("复制项目到 Colab 本地磁盘 (首次约 1-3 分钟)...")
    t0 = time.time()

    def ignore_large(d, files):
        skip = set()
        for f in files:
            full = os.path.join(d, f)
            if os.path.isfile(full) and os.path.getsize(full) > 200_000_000:
                skip.add(f)
            if f in ('.git', '__pycache__', 'outputs', 'glove.6B.zip'):
                skip.add(f)
        return skip

    shutil.copytree(DRIVE_PROJECT, PROJECT_DIR, ignore=ignore_large)
    print(f"复制完成 ({time.time()-t0:.1f}s)")
else:
    print(f"项目已存在: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"工作目录: {os.getcwd()}")

In [ ]:
# ============================================================
# 0.4 复制主数据集 CSV (组3 需要)
# ============================================================
os.makedirs(os.path.join(PROJECT_DIR, 'News _dataset'), exist_ok=True)

data_files = [
    ('fakenews 2.csv',
     os.path.join(DRIVE_PROJECT, 'fakenews 2.csv'),
     os.path.join(PROJECT_DIR, 'fakenews 2.csv')),
    ('Fake.csv',
     os.path.join(DRIVE_PROJECT, 'News _dataset', 'Fake.csv'),
     os.path.join(PROJECT_DIR, 'News _dataset', 'Fake.csv')),
    ('True.csv',
     os.path.join(DRIVE_PROJECT, 'News _dataset', 'True.csv'),
     os.path.join(PROJECT_DIR, 'News _dataset', 'True.csv')),
]

for name, src, dst in data_files:
    if os.path.exists(dst):
        print(f"  OK  {name} ({os.path.getsize(dst)/1e6:.1f} MB)")
    elif os.path.exists(src):
        t0 = time.time()
        shutil.copy2(src, dst)
        print(f"  Copied {name} ({os.path.getsize(dst)/1e6:.1f} MB, {time.time()-t0:.1f}s)")
    else:
        print(f"  MISSING: {src}")

print("\n数据准备完成")

In [ ]:
# ============================================================
# 0.5 安装依赖
# ============================================================
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn tqdm
!pip install -q transformers accelerate

# 组3: 网搜 + NLI
!pip install -q duckduckgo_search

# 组5: MMDFND 额外依赖
!pip install -q timm positional_encodings cn_clip

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("\n依赖安装完成")

In [ ]:
# ============================================================
# 0.6 验证关键文件
# ============================================================
required = [
    'experiments/config.py',
    'experiments/metrics.py',
    'experiments/group3_multiagent/agents.py',
    'experiments/group3_multiagent/orchestrator.py',
    'experiments/group3_multiagent/search_tools.py',
    'experiments/group3_multiagent/run_group3.py',
    'experiments/group5_mmdfnd/run_group5.py',
    'experiments/group5_mmdfnd/adapter.py',
    'MMDFND/main.py',
    'MMDFND/run.py',
]
for f in required:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  [{status}] {f}")

---
## 1. 实验组 3：多智能体网络搜索验证

**Pipeline**：`ClaimExtractor` → `WebSearcher` → `EvidenceScorer (MNLI)` → `JudgeAgent`

- **主张抽取**：优先用 LLM（OpenAI / 本地 flan-t5），无 API 则用启发式句子切分
- **网搜**：Tavily → SerpAPI → DuckDuckGo → Mock 自动降级
- **NLI 打分**：`roberta-large-mnli`，标准三分类（premise=证据, hypothesis=主张）
- **判定**：rule-based 或 LLM judge

**无需任何 API key** 即可运行（本地 LLM + DuckDuckGo）。若有 `OPENAI_API_KEY`/`TAVILY_API_KEY` 效果更好。

In [ ]:
# ============================================================
# 1.1 (可选) 设置 API Key — 不设也能跑
# ============================================================
import os

# 如果你有 key，取消注释并填入：
# os.environ['OPENAI_API_KEY'] = 'sk-...'     # 用 GPT 抽取主张 + 判定
# os.environ['TAVILY_API_KEY'] = 'tvly-...'   # 用 Tavily 搜索

print(f"OPENAI_API_KEY: {'已设置' if os.environ.get('OPENAI_API_KEY') else '未设置 (用本地 flan-t5)'}")
print(f"TAVILY_API_KEY: {'已设置' if os.environ.get('TAVILY_API_KEY') else '未设置 (用 DuckDuckGo)'}")

In [ ]:
# ============================================================
# 1.2 运行组 3（rule-based judge，默认 100 样本）
# ============================================================
%cd /content/fakenews-detector

# --quick: 20 样本快速验证；去掉 --quick 跑完整 100 样本
!python -m experiments.group3_multiagent.run_group3 --quick --judge-mode rule

In [ ]:
# ============================================================
# 1.3 (可选) LLM Judge 模式
# ============================================================
# 需要 OpenAI API key 或本地 LLM
# !python -m experiments.group3_multiagent.run_group3 --quick --judge-mode llm

In [ ]:
# ============================================================
# 1.4 查看组3结果
# ============================================================
import pandas as pd
from IPython.display import display, Image as IPImage

g3_csv = 'outputs/group3_multiagent/results.csv'
if os.path.exists(g3_csv):
    df = pd.read_csv(g3_csv)
    display(df[['experiment_name', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc']]
            .sort_values('f1', ascending=False)
            .style.format({'accuracy': '{:.3f}', 'precision': '{:.3f}',
                           'recall': '{:.3f}', 'f1': '{:.3f}', 'auc_roc': '{:.3f}'}))
else:
    print("组3 尚未运行")

g3_fig = 'outputs/group3_multiagent/group3_comparison.png'
if os.path.exists(g3_fig):
    display(IPImage(g3_fig, width=700))

In [ ]:
# ============================================================
# 1.5 手动测试单篇文章
# ============================================================
from experiments.config import ExperimentConfig, MultiAgentConfig
from experiments.group3_multiagent.orchestrator import VerificationPipeline

cfg = ExperimentConfig(multiagent=MultiAgentConfig(judge_mode='rule'))
pipeline = VerificationPipeline.from_config(cfg.multiagent)

test_article = """
Scientists at MIT have discovered a new material that can conduct electricity 
with zero resistance at room temperature. The breakthrough, published in Nature, 
could revolutionize energy transmission and computing.
"""

result = pipeline.verify(test_article.strip())

print(f"Prediction: {'REAL' if result.prediction == 1 else 'FAKE'}")
print(f"Confidence: {result.confidence:.2f}")
print(f"Reasoning:  {result.reasoning}")
print(f"Claims extracted: {len(result.claims)}")
for c in result.claims:
    print(f"  - {c.text[:100]}")
print(f"\nNLI scores: {len(result.nli_scores)}")
for s in result.nli_scores[:5]:
    print(f"  E={s.entailment:.2f}  C={s.contradiction:.2f}  N={s.neutral:.2f}  | {s.evidence[:60]}...")

---
## 2. 实验组 5：MMDFND 论文复现

**MMDFND** = Multi-Modal Domain-aware Fake News Detection（多模态领域感知虚假新闻检测）

- 数据：**Weibo 微博数据集**（中文 CSV + 配图）
- 架构：Chinese RoBERTa (文本) + Chinese-CLIP (图文) + MAE (图像) + PLE 专家门控 + Pivot Transformer

### 前置准备清单

| 资源 | 放置位置 | 必需？ |
|------|----------|:------:|
| `train_origin.csv`, `val_origin.csv`, `test_origin.csv` | `data 3/` (已在 Drive) | **是** |
| `nonrumor_images/`, `rumor_images/` | `MMDFND/data/` | **是** |
| Chinese RoBERTa | `MMDFND/pretrained_model/chinese_roberta_wwm_base_ext_pytorch/` | **是** |
| `mae_pretrain_vit_base.pth` | `MMDFND/` | **是** |
| `clip_cn_vit-b-16.pt` | `MMDFND/` | **是** |
| `*_loader.pkl` (6个) | `MMDFND/data/` | 预处理生成 |

In [ ]:
# ============================================================
# 2.1 环境检查
# ============================================================
%cd /content/fakenews-detector
!python -m experiments.group5_mmdfnd.run_group5 --check

In [ ]:
# ============================================================
# 2.2 复制/链接数据到 MMDFND/data/
#     Drive 上的 data 3/ 含 CSV，图片需单独放入
# ============================================================
import os, shutil

MMDFND_DATA = os.path.join(PROJECT_DIR, 'MMDFND', 'data')
os.makedirs(MMDFND_DATA, exist_ok=True)

# ---- 复制 CSV ----
DRIVE_DATA3 = '/content/drive/MyDrive/data 3'
for csv_name in ['train_origin.csv', 'val_origin.csv', 'test_origin.csv']:
    src = os.path.join(DRIVE_DATA3, csv_name)
    dst = os.path.join(MMDFND_DATA, csv_name)
    if os.path.exists(dst):
        print(f"  OK  {csv_name}")
    elif os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied {csv_name}")
    else:
        print(f"  MISSING  {src}")

# ---- 图片目录 ----
# 如果图片也在 Drive，修改下面的路径：
# DRIVE_IMAGES = '/content/drive/MyDrive/weibo_images'
# for img_dir in ['nonrumor_images', 'rumor_images']:
#     src = os.path.join(DRIVE_IMAGES, img_dir)
#     dst = os.path.join(MMDFND_DATA, img_dir)
#     if not os.path.exists(dst) and os.path.exists(src):
#         os.symlink(src, dst)  # 符号链接，省空间
#         print(f"  Linked {img_dir}")

for d in ['nonrumor_images', 'rumor_images']:
    p = os.path.join(MMDFND_DATA, d)
    if os.path.exists(p):
        n = len(os.listdir(p))
        print(f"  OK  {d}/ ({n} files)")
    else:
        print(f"  MISSING  {d}/  ← 需要把微博图片放到 MMDFND/data/{d}/")

In [ ]:
# ============================================================
# 2.3 下载/放置预训练模型
# ============================================================
import os

MMDFND_ROOT = os.path.join(PROJECT_DIR, 'MMDFND')

checks = {
    'Chinese RoBERTa': os.path.join(MMDFND_ROOT, 'pretrained_model',
                                     'chinese_roberta_wwm_base_ext_pytorch'),
    'MAE ViT-base':    os.path.join(MMDFND_ROOT, 'mae_pretrain_vit_base.pth'),
    'Chinese-CLIP':    os.path.join(MMDFND_ROOT, 'clip_cn_vit-b-16.pt'),
}

for name, path in checks.items():
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING'}  {name}")
    if not exists:
        print(f"         → 需放到: {path}")

print("\n如果有 MISSING，请参考 MMDFND README 下载对应权重文件。")
print("Chinese RoBERTa: https://drive.google.com/drive/folders/1y2k22iMG1i1f302NLf-bj7UEe9zwTwLR")
print("MAE:  https://github.com/facebookresearch/mae")
print("CLIP: https://github.com/OFA-Sys/Chinese-CLIP")

In [ ]:
# ============================================================
# 2.4 数据预处理 — 生成 *_loader.pkl
#     必须在 MMDFND/ 目录下运行
#     前提：CSV + 图片目录 + 预训练模型都已就位
# ============================================================
%cd /content/fakenews-detector/MMDFND

import os
pkls = ['data/train_loader.pkl', 'data/train_clip_loader.pkl',
        'data/val_loader.pkl',   'data/val_clip_loader.pkl',
        'data/test_loader.pkl',  'data/test_clip_loader.pkl']

all_exist = all(os.path.exists(p) for p in pkls)

if all_exist:
    print("所有 pkl 已存在，跳过预处理。")
    for p in pkls:
        print(f"  OK  {p} ({os.path.getsize(p)/1e6:.1f} MB)")
else:
    print("开始预处理（需要图片目录 + CSV），可能需要 10-30 分钟...")
    !python data_pre.py
    print("--- data_pre.py 完成 ---")
    !python clip_data_pre.py
    print("--- clip_data_pre.py 完成 ---")
    for p in pkls:
        status = 'OK' if os.path.exists(p) else 'FAILED'
        print(f"  [{status}] {p}")

In [ ]:
# ============================================================
# 2.5 训练 MMDFND
#     --dataset weibo  (对应 data 3/ 里的 CSV)
#     --root_path      (CSV 根目录，已改代码支持外部路径)
# ============================================================
%cd /content/fakenews-detector/MMDFND

# 方式 A：直接用 MMDFND/main.py
!python main.py --dataset weibo --root_path "/content/fakenews-detector/MMDFND/data/" --epoch 50

# 方式 B：用 experiments wrapper（等价）
# %cd /content/fakenews-detector
# !python -m experiments.group5_mmdfnd.run_group5 --train --dataset weibo \
#     --data-root "/content/fakenews-detector/MMDFND/data/" --epochs 50

In [ ]:
# ============================================================
# 2.6 查看训练结果
# ============================================================
import os
MMDFND_ROOT = os.path.join(PROJECT_DIR, 'MMDFND')

param_file = os.path.join(MMDFND_ROOT, 'param_model', 'MMDFND', 'parameter_mmdfnd.pkl')
if os.path.exists(param_file):
    print(f"模型已保存: {param_file} ({os.path.getsize(param_file)/1e6:.1f} MB)")
else:
    print("模型文件未找到，训练可能未完成。")
    # 列出 param_model 下所有文件
    pm_dir = os.path.join(MMDFND_ROOT, 'param_model')
    if os.path.isdir(pm_dir):
        for root, dirs, files in os.walk(pm_dir):
            for f in files:
                fp = os.path.join(root, f)
                print(f"  {fp} ({os.path.getsize(fp)/1e6:.1f} MB)")

---
## 3. 结果对比 & 保存

In [ ]:
# ============================================================
# 3.1 汇总结果
# ============================================================
import pandas as pd
from IPython.display import display

frames = []
for name, path in [('Group3', 'outputs/group3_multiagent/results.csv')]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        frames.append(df)
        print(f"Loaded {name}: {len(df)} rows")

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    cols = ['experiment_name', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc']
    cols = [c for c in cols if c in all_df.columns]
    display(all_df[cols].sort_values('f1', ascending=False)
            .style.format({c: '{:.3f}' for c in cols if c != 'experiment_name'}))
else:
    print("没有找到结果文件")

In [ ]:
# ============================================================
# 3.2 保存结果回 Drive
# ============================================================
import shutil

DRIVE_OUTPUT = os.path.join(DRIVE_PROJECT, 'outputs')
LOCAL_OUTPUT = os.path.join(PROJECT_DIR, 'outputs')

if os.path.isdir(LOCAL_OUTPUT):
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    for sub in ['group3_multiagent', 'group5_mmdfnd']:
        src = os.path.join(LOCAL_OUTPUT, sub)
        dst = os.path.join(DRIVE_OUTPUT, sub)
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  Saved {sub}/ -> Drive")

    # MMDFND 训练权重
    param_src = os.path.join(PROJECT_DIR, 'MMDFND', 'param_model')
    param_dst = os.path.join(DRIVE_PROJECT, 'MMDFND', 'param_model')
    if os.path.isdir(param_src):
        os.makedirs(os.path.dirname(param_dst), exist_ok=True)
        if os.path.isdir(param_dst):
            shutil.rmtree(param_dst)
        shutil.copytree(param_src, param_dst)
        print(f"  Saved MMDFND/param_model/ -> Drive")

    print("\n结果已保存到 Google Drive")
else:
    print("本地 outputs/ 目录不存在")